# Similaridade entre topicos da pauta e do discurso (BERTopic)
Este notebook gera visualizacoes comparando a similaridade entre topicos da pauta e topicos do discurso antes e depois da eleicao, por partido, para os resultados gerados pelo BERTopic.

In [9]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / "data" / "party_agenda" / "embeddings"
PARTIES = ["MDB", "NOVO", "PL", "PSOL", "PT", "UNIAO"]
ELECTION_PERIODS = ["antesDaEleicao", "depoisDaEleicao"]
CSV_NAME = "similaridade_topics_discurso_topics_agenda.csv"

def load_similarity_table(party: str, period: str) -> pd.DataFrame:
    file_path = DATA_DIR / party / "similaridade" / "topics" / "bertopic" / period / CSV_NAME
    if not file_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(file_path)
    df["party"] = party
    df["period"] = period
    return df

rows = []
for party in PARTIES:
    for period in ELECTION_PERIODS:
        rows.append(load_similarity_table(party, period))

similarity_df = pd.concat([df for df in rows if not df.empty], ignore_index=True)
similarity_df.head()

,agenda_topic,agenda_terms,discourse_topic,discourse_terms,cosine_similarity,party,period
0,0,"0.656*""caminho brasil país"" + 0.638*""pensar br...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.630360,MDB,antesDaEleicao
1,1,"0.560*""sustentar"" + 0.558*""responsabilidade fi...",2,"0.014*""mdb"" + 0.007*""orador"" + 0.007*""revisão""...",0.229074,MDB,antesDaEleicao
2,2,"0.601*""sustentabilidade"" + 0.577*""sustentável""...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.170035,MDB,antesDaEleicao
3,3,"0.555*""inclusão"" + 0.548*""possibilidade"" + 0.5...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.123234,MDB,antesDaEleicao
4,4,"0.619*""sustentabilidade"" + 0.584*""sustentável""...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.544159,MDB,antesDaEleicao


In [10]:
if similarity_df.empty:
    raise ValueError("Nenhum arquivo foi encontrado. Verifique os caminhos dos CSVs.")

similarity_df["cosine_similarity"] = pd.to_numeric(similarity_df["cosine_similarity"], errors="coerce")
summary = (
    similarity_df
    .groupby(["party", "period"], as_index=False)
    .agg(
        mean_similarity=("cosine_similarity", "mean"),
        median_similarity=("cosine_similarity", "median"),
        max_similarity=("cosine_similarity", "max"),
        rows=("cosine_similarity", "size"),
    )
)
summary

,party,period,mean_similarity,median_similarity,max_similarity,rows
0,MDB,antesDaEleicao,0.401346,0.514998,0.630360,9
1,MDB,depoisDaEleicao,0.420800,0.476170,0.702839,9
2,NOVO,antesDaEleicao,0.561439,0.561439,0.626272,2
3,NOVO,depoisDaEleicao,0.512328,0.512328,0.554470,2
4,PL,antesDaEleicao,0.338272,0.332210,0.436078,7
5,PL,depoisDaEleicao,0.287722,0.297335,0.458846,7
6,PSOL,antesDaEleicao,0.399751,0.351411,0.649196,7
7,PSOL,depoisDaEleicao,0.275833,0.277954,0.336012,7
8,PT,antesDaEleicao,0.604250,0.532571,0.765851,3
9,PT,depoisDaEleicao,0.499476,0.505766,0.648994,3


In [11]:
fig = px.bar(
    summary,
    x="party",
    y="mean_similarity",
    color="period",
    barmode="group",
    title="Similaridade media entre topicos de pauta e discurso (BERTopic)",
    labels={"mean_similarity": "Similaridade media", "period": "Periodo"},
    text=summary["mean_similarity"].round(3).astype(str),
)
fig.update_layout(height=420, legend_title_text="Periodo")
fig.update_traces(textposition="outside")
fig.show()

In [12]:
# Tabela comparativa dos topicos mais similares
top_matches = (
    similarity_df
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    .groupby(["party", "period"], as_index=False)
    .head(5)
    .reset_index(drop=True)
)
top_matches.head()

,agenda_topic,agenda_terms,discourse_topic,discourse_terms,cosine_similarity,party,period
0,0,"0.656*""caminho brasil país"" + 0.638*""pensar br...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.630360,MDB,antesDaEleicao
1,7,"0.617*""caminho brasil principal"" + 0.581*""bras...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.563163,MDB,antesDaEleicao
2,8,"0.749*""precisar pensar brasil"" + 0.739*""pensar...",1,"0.014*""mdb"" + 0.008*""brasil"" + 0.006*""grande"" ...",0.545867,MDB,antesDaEleicao
3,4,"0.619*""sustentabilidade"" + 0.584*""sustentável""...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.544159,MDB,antesDaEleicao
4,6,"0.600*""caminho brasil principal"" + 0.592*""bras...",3,"0.008*""recurso"" + 0.008*""povo"" + 0.007*""real"" ...",0.514998,MDB,antesDaEleicao


In [13]:
fig = px.scatter(
    top_matches,
    x="agenda_topic",
    y="discourse_topic",
    size="cosine_similarity",
    color="period",
    facet_row="party",
    title="Topicos mais similares entre pauta e discurso - BERTopic (top 5 por partido/periodo)",
    labels={"agenda_topic": "Topico de pauta", "discourse_topic": "Topico de discurso"},
    hover_data=["cosine_similarity", "agenda_terms", "discourse_terms"],
)
fig.update_layout(height=220 * len(PARTIES))
fig.show()

In [14]:
# Topicos do UNIAO — antes e depois da eleicao
uniao = (
    similarity_df[similarity_df["party"] == "UNIAO"]
    .sort_values(["period", "cosine_similarity"], ascending=[True, False])
    [["period", "agenda_topic", "agenda_terms", "discourse_topic", "discourse_terms", "cosine_similarity"]]
)
if uniao.empty:
    raise ValueError("Sem dados para UNIAO.")

import re as _re

def _format_terms(terms_str: str) -> str:
    """Extract term labels from weighted string; return bullet list."""
    matches = _re.findall(r'\d+\.\d+\*"([^"]*)"', str(terms_str))
    terms = [t.strip() for t in matches if t.strip()]
    if terms:
        return "\n".join(f"\u2022 {t.strip()}" for t in terms[:10])
    return str(terms_str)

_disp = uniao.copy()
_disp["cosine_similarity"] = _disp["cosine_similarity"].round(3)
_disp["agenda_terms"] = _disp["agenda_terms"].apply(_format_terms)
_disp["discourse_terms"] = _disp["discourse_terms"].apply(_format_terms)
_disp.columns = ['Período', 'Tóp. Pauta', 'Termos Pauta', 'Tóp. Discurso', 'Termos Discurso', 'Similaridade']

_n = len(_disp)
_n_cols = len(_disp.columns)
_row_colors = ["#f2f4f8" if i % 2 == 0 else "white" for i in range(_n)]

fig_uniao = go.Figure(
    data=[
        go.Table(
            columnwidth=[100, 60, 310, 60, 310, 80],
            header=dict(
                values=list(_disp.columns),
                fill_color="#2c3e50",
                font=dict(color="white", size=12, family="Arial"),
                align=["left", "center", "left", "center", "left", "center"],
                height=38,
                line=dict(color="#2c3e50", width=1),
            ),
            cells=dict(
                values=[_disp[col] for col in _disp.columns],
                fill_color=[_row_colors] * _n_cols,
                align=["left", "center", "left", "center", "left", "center"],
                font=dict(size=11, family="Arial"),
                height=180,
                line=dict(color="#ced4da", width=1),
            ),
        )
    ]
)
fig_uniao.update_layout(
    title=dict(text="Tópicos de pauta vs. discurso — UNIAO (BERTopic)", font=dict(size=14, family="Arial")),
    height=max(560, 38 + _n * 180 + 60),
    margin=dict(l=8, r=8, t=60, b=8),
)
fig_uniao.show()

In [15]:
# Tabela com topicos mais similares para todos os partidos

def render_topic_table(df: pd.DataFrame, title: str) -> None:
    if df.empty:
        raise ValueError(f"Sem dados para {title}.")
    df = df.copy()
    df["cosine_similarity"] = df["cosine_similarity"].round(3)
    fig = go.Figure(
        data=[
            go.Table(
                header=dict(values=list(df.columns), fill_color="#e9ecef", align="left"),
                cells=dict(values=[df[col] for col in df.columns], align="left"),
            )
        ]
    )
    fig.update_layout(title=title, height=420)
    fig.show()

all_parties = (
    top_matches
    .sort_values(["party", "period", "cosine_similarity"], ascending=[True, True, False])
    [["party", "period", "agenda_topic", "discourse_topic", "cosine_similarity"]]
)

render_topic_table(all_parties, "Topicos mais similares por partido - BERTopic")

In [16]:
# Tabela resumida para exportacao ou inspecao
summary_table = summary.sort_values(["party", "period"])
summary_table

,party,period,mean_similarity,median_similarity,max_similarity,rows
0,MDB,antesDaEleicao,0.401346,0.514998,0.630360,9
1,MDB,depoisDaEleicao,0.420800,0.476170,0.702839,9
2,NOVO,antesDaEleicao,0.561439,0.561439,0.626272,2
3,NOVO,depoisDaEleicao,0.512328,0.512328,0.554470,2
4,PL,antesDaEleicao,0.338272,0.332210,0.436078,7
5,PL,depoisDaEleicao,0.287722,0.297335,0.458846,7
6,PSOL,antesDaEleicao,0.399751,0.351411,0.649196,7
7,PSOL,depoisDaEleicao,0.275833,0.277954,0.336012,7
8,PT,antesDaEleicao,0.604250,0.532571,0.765851,3
9,PT,depoisDaEleicao,0.499476,0.505766,0.648994,3
